# Cyber Security Attack Analysis
**Course:** Data Visualization — Team 04

This notebook runs the full workflow specified in the project proposal:
1. **Data Profiling, Cleaning & Validation** (`Datasets/cybersecurity_attacks.csv` → `Datasets/cleaned_cybersecurity_attacks.csv`)
2. **Interactive Plotly Visualizations** (10 charts using the colorblind-safe Okabe-Ito palette & ordered light-to-dark severity scale)
3. **Static Exports** (10 high-DPI PNGs + combined multi-page PDF in `exports/`)
4. **Interactive KPI Dashboard** (`dashboard.html` with Attack Type, Severity, and Year filters)
5. **Key Findings Verification** (Section 6 summary statistics)

In [ ]:
import pandas as pd
import plotly.express as px

import analysis

print("Project directory:", analysis.BASE)

## 1. Data Profiling, Cleaning & Validation
- Recodes the 5 missing-value columns into `Yes`/`No` detection flags without imputation.
- Parses `Timestamp`, `Geo-location Data` (`City`, `State`), and `Device Information` (`Device OS`, `Browser Family`).
- Re-checks duplicates after type conversions, validates numeric ranges, adds derived fields (`High Anomaly`, `Anomaly Band`, `Packet Size Band`), and drops privacy-sensitive fields.

In [ ]:
clean = analysis.clean_dataset()
clean.head()

## 2. Generate Interactive Charts, Static PNG/PDF Exports & Dashboard
Writes:
- `charts/*.html` (10 interactive Plotly charts)
- `exports/*.png` & `exports/cybersecurity_attack_charts.pdf`
- `dashboard.html` (interactive KPI row + Attack Type, Severity, and Year filters)

In [ ]:
analysis.build_plotly_charts(clean)
analysis.build_static_exports(clean)
analysis.build_interactive_dashboard(clean)

## 3. Inline Preview — Key Finding: Response Action by Severity

In [ ]:
act = pd.crosstab(clean["Severity Level"], clean["Action Taken"], normalize="index").mul(100).round(1)
act_long = act.reset_index().melt(id_vars="Severity Level", var_name="Action Taken", value_name="Percent")
fig = px.bar(
    act_long,
    x="Severity Level",
    y="Percent",
    color="Action Taken",
    barmode="group",
    category_orders={"Severity Level": analysis.SEV_ORDER, "Action Taken": analysis.ACTION_ORDER},
    color_discrete_map=analysis.ACTION_COLORS,
    text="Percent",
    labels={"Percent": "Share of Events (%)"},
    title="Key Finding: Blocking rate (~34%) does not increase for High-severity attacks",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_yaxes(range=[0, 44], ticksuffix="%")
analysis.style_plotly(fig, show_legend=True).show()

## 4. Section 6 Key Findings Summary

In [ ]:
analysis.print_summary(clean)